In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

: 

In [ ]:
!pip install quandl
import quandl
!pip install yfinance
import yfinance as yf

In [ ]:
data = yf.download("TATACONSUM.NS", start="2010-01-01", end="2025-01-01")

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(data['Close'], color='blue', label='Closing Price')
plt.title('Tata Consumer Products Ltd (Closing Price Over Time)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Close Price (INR)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
data.head(10)

In [ ]:
data['Open - Close'] = data['Open'] - data['Close']
data['High - Low'] = data['High'] - data['Low']
data = data.dropna()

In [ ]:
x = data[['Open - Close', 'High - Low']]
x.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(data[('Open', 'TATACONSUM.NS')] - data[('Close', 'TATACONSUM.NS')], 
         label='Open - Close', color='orange')
plt.plot(data[('High', 'TATACONSUM.NS')] - data[('Low', 'TATACONSUM.NS')], 
         label='High - Low', color='green')

plt.title('Feature Trends (Open-Close & High-Low)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price Difference (INR)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
Y= np.where(data['Close'].shift(-1)>data['Close'],1,-1)

In [ ]:
Y

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, Y, test_size=0.25, random_state = 44)
y_train = y_train.ravel()
y_test = y_test.ravel()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

#using gridsearch to find the best parameter
params = {'n_neighbors':[2,3,4,5,6,7,8,9,10,11,12,13,14,15]}
knn = neighbors.KNeighborsClassifier()
model = GridSearchCV(knn, params, cv=5)

#fit the model
model.fit(X_train, y_train)

#Accuracy score
accuracy_train = accuracy_score(y_train, model.predict(X_train))
accuracy_test = accuracy_score(y_test, model.predict(X_test))

print ('Train_data Accuracy: %.2f' %accuracy_train)
print ('Test_data Accuracy: %.2f' %accuracy_test)

In [ ]:
predictions_classification = model.predict(X_test)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(y_test[:100], label='Actual Trend', color='green')
plt.plot(predictions_classification[:100], label='Predicted Trend', color='red', linestyle='--')
plt.title('Predicted vs Actual Stock Movement (Up=1, Down=-1)')
plt.xlabel('Days (sample of 100)')
plt.ylabel('Trend')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
actual_predicted_data = pd.DataFrame({'Actual Class':y_test, 'Predicted Class':predictions_classification})

In [ ]:
actual_predicted_data.head(10)

In [ ]:
y = data['Close']
y

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn import neighbors
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(x, y, test_size=0.25, random_state=44)

#using gridsearch to find the best parameter
params = {'n_neighbors': [2,3,4,5,6,7,8,9,10,11,12,13,14,15]}
knn_reg = neighbors.KNeighborsRegressor()
model_reg = GridSearchCV(knn_reg, params, cv=5)

#fit the model and make predictions
model_reg.fit(X_train_reg,y_train_reg)
predictions = model_reg.predict(X_test_reg)

In [ ]:
print(predictions)

In [ ]:
#rmse
rms=np.sqrt(np.mean(np.power((np.array(y_test)-np.array(predictions)),2)))
float(rms)

In [ ]:
valid = pd.DataFrame({
    'Actual Close': y_test_reg.to_numpy().flatten(),
    'Predicted Close Value': predictions.to_numpy().flatten() if hasattr(predictions, 'to_numpy') else predictions.flatten()
})
valid.head(10)


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14,10))

# Plot all (smoothed)
ax[0].plot(valid['Actual Close'].rolling(5).mean(), label='Actual (Smoothed)', color='blue', linewidth=2)
ax[0].plot(valid['Predicted Close Value'].rolling(5).mean(), label='Predicted (Smoothed)', color='red', linestyle='--', linewidth=2)
ax[0].set_title('All Test Data (Smoothed)')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# Plot last 100 samples (zoomed)
ax[1].plot(valid['Actual Close'].values[-100:], label='Actual', color='blue', linewidth=2)
ax[1].plot(valid['Predicted Close Value'].values[-100:], label='Predicted', color='red', linestyle='--', linewidth=2)
ax[1].set_title('Zoomed: Last 100 Samples')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()